In [1]:
import pandas as pd

> bank.csv download link
[Bank Marketing Dataset](https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset)

In [2]:
# Load the csv.
df = pd.read_csv("bank.csv")
# Export as .xlsx file.
df.to_excel("bank.xlsx", index=False)

# ⚙️ Preprocessing

## 🗑️ removing unnecessary features.

In [3]:
unnecessary_features = [
    'age', 'default', 'contact', 'day', 'month', 'duration',
    'campaign', 'pdays', 'previous', 'poutcome', 'deposit'
]
df = df.drop(columns=unnecessary_features)

In [4]:
df.head(10)

,job,marital,education,balance,housing,loan
0,admin.,married,secondary,2343,yes,no
1,admin.,married,secondary,45,no,no
2,technician,married,secondary,1270,yes,no
3,services,married,secondary,2476,yes,no
4,admin.,married,tertiary,184,no,no
5,management,single,tertiary,0,yes,yes
6,management,married,tertiary,830,yes,yes
7,retired,divorced,secondary,545,yes,no
8,technician,married,secondary,1,yes,no
9,services,single,secondary,5090,yes,no


## 🔙 backup taken.

In [5]:
df_bak = df.copy()

## 📇 Job

In [6]:
print(df['job'].nunique())

12


In [7]:
print(df['job'].value_counts())

job
management       2566
blue-collar      1944
technician       1823
admin.           1334
services          923
retired           778
self-employed     405
student           360
unemployed        357
entrepreneur      328
housemaid         274
unknown            70
Name: count, dtype: int64


In [8]:
df['job'].sample(10)

7770     blue-collar
2176      technician
4498      management
10251       services
4780     blue-collar
10224     technician
6493      technician
2358      management
4434      management
2309        services
Name: job, dtype: object

### 💼 JOB => stability score

In [9]:
df1 = df.copy()

In [10]:
job_stability_map = {
    'management': 0.9,
    'technician': 0.85,
    'admin.': 0.8,
    'services': 0.65,
    'blue-collar': 0.6,
    'self-employed': 0.55,
    'entrepreneur': 0.5,
    'housemaid': 0.45,
    'retired': 0.7,
    'student': 0.3,
    'unemployed': 0.1,
    'unknown': 0.5
}
print(len(job_stability_map))

12


In [11]:
df1["job_stability_score"] = df1["job"].map(job_stability_map).fillna(0.5)
df1.drop(columns=["job"], inplace=True)

In [12]:
df1['job_stability_score'].sample(7)

4016     0.70
4631     0.50
5427     0.85
9023     0.60
11161    0.85
8585     0.90
3753     0.90
Name: job_stability_score, dtype: float64

In [13]:
df1.columns

Index(['marital', 'education', 'balance', 'housing', 'loan',
       'job_stability_score'],
      dtype='object')

## 💍 Marital status

In [14]:
df2 = df1.copy()

In [15]:
print(df2['marital'].value_counts())

marital
married     6351
single      3518
divorced    1293
Name: count, dtype: int64


In [16]:
df2 = pd.get_dummies(df2, columns=['marital'], dtype=int)

In [17]:
print(df2.columns)

Index(['education', 'balance', 'housing', 'loan', 'job_stability_score',
       'marital_divorced', 'marital_married', 'marital_single'],
      dtype='object')


In [18]:
marital = ['marital_divorced', 'marital_married', 'marital_single']

In [19]:
df2[marital].head(10)

,marital_divorced,marital_married,marital_single
0,0,1,0
1,0,1,0
2,0,1,0
3,0,1,0
4,0,1,0
5,0,0,1
6,0,1,0
7,1,0,0
8,0,1,0
9,0,0,1


In [20]:
df2.columns

Index(['education', 'balance', 'housing', 'loan', 'job_stability_score',
       'marital_divorced', 'marital_married', 'marital_single'],
      dtype='object')

In [21]:
subset = set(df1.columns)
sprset = set(df2.columns)

print(subset.issubset(sprset))
missing_cols = subset - sprset
print(missing_cols)

False
{'marital'}


## 🎓 Educational qualification

In [22]:
df3 = df2.copy()

In [23]:
print(df3['education'].value_counts())

education
secondary    5476
tertiary     3689
primary      1500
unknown       497
Name: count, dtype: int64


In [24]:
df3 = pd.get_dummies(df3, columns=["education"],  dtype=int)

In [25]:
print(df3.columns)

Index(['balance', 'housing', 'loan', 'job_stability_score', 'marital_divorced',
       'marital_married', 'marital_single', 'education_primary',
       'education_secondary', 'education_tertiary', 'education_unknown'],
      dtype='object')


In [26]:
education = [
    'education_primary',
    'education_secondary',
    'education_tertiary',
    'education_unknown'
]
df3[education].sample(5)

,education_primary,education_secondary,education_tertiary,education_unknown
10225,0,1,0,0
10066,0,1,0,0
10640,0,1,0,0
6789,0,0,1,0
6205,0,0,1,0


In [27]:
subset = set(df2.columns)
sprset = set(df3.columns)

print(subset.issubset(sprset))
missing_cols = subset - sprset
print(missing_cols)

False
{'education'}


## 💸 Balance

In [28]:
df4 = df3.copy()

In [29]:
df4.rename(columns={'balance': 'avg_balance'}, inplace=True)

In [30]:
max_avg_bal = df4['avg_balance'].max()
min_avg_bal = df4['avg_balance'].min()

In [31]:
print("Maximum balance: ", max_avg_bal, "\nMinimum balance: ", min_avg_bal)

Maximum balance:  81204 
Minimum balance:  -6847


### 📝 Nomalization of Average Balance.

In [32]:
df4["balance_norm"] = (df4["avg_balance"] - (min_avg_bal)) / (max_avg_bal - (min_avg_bal))

In [33]:
print(df4["balance_norm"].head(6))

0    0.104371
1    0.078273
2    0.092185
3    0.105882
4    0.079851
5    0.077762
Name: balance_norm, dtype: float64


In [34]:
df4.drop(columns=["avg_balance"], inplace=True)

In [35]:
subset = set(df3.columns)
sprset = set(df4.columns)

print(subset.issubset(sprset))
missing_cols = subset - sprset
print("Removed features: ", missing_cols)
added_features = sprset - subset
print("Added features: ", added_features)

False
Removed features:  {'balance'}
Added features:  {'balance_norm'}


## 🏠💰 Housing loan and Other loans

In [36]:
df5 = df4.copy()

In [37]:
df5['housing'].value_counts()

housing
no     5881
yes    5281
Name: count, dtype: int64

In [38]:
loans = [
    'housing',
    'loan'
]

In [39]:
df5[loans] = df5[loans].replace({'yes': 1, 'no': 0})

/var/folders/f3/6s9_42cn1wd8b4klch2s27k80000gn/T/ipykernel_23478/2031821274.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df5[loans] = df5[loans].replace({'yes': 1, 'no': 0})


In [40]:
df5[loans].head(10)

,housing,loan
0,1,0
1,0,0
2,1,0
3,1,0
4,0,0
5,1,1
6,1,1
7,1,0
8,1,0
9,1,0


In [41]:
subset = set(df4.columns)
sprset = set(df5.columns)

print(subset.issubset(sprset))
missing_cols = subset - sprset
print("Removed features: ", missing_cols)
added_features = sprset - subset
print("Added features: ", added_features)

True
Removed features:  set()
Added features:  set()


# ✍🏻 New intoduced features

In [42]:
import numpy as np

## 💳 Financial Features

In [43]:
df6 = df5.copy()

In [44]:
df6["monthly_income_avg"] = np.random.randint(8000, 60000, len(df6))
df6["annual_income"] = df6["monthly_income_avg"] * 12

df6["income_variance"] = np.random.uniform(0.1, 0.7, len(df6))

In [45]:
print(df6.columns)

Index(['housing', 'loan', 'job_stability_score', 'marital_divorced',
       'marital_married', 'marital_single', 'education_primary',
       'education_secondary', 'education_tertiary', 'education_unknown',
       'balance_norm', 'monthly_income_avg', 'annual_income',
       'income_variance'],
      dtype='object')


In [46]:
financial_features = ['monthly_income_avg', 'annual_income', 'income_variance']

In [47]:
df6[financial_features].head(10)

,monthly_income_avg,annual_income,income_variance
0,52561,630732,0.386010
1,19071,228852,0.157927
2,44924,539088,0.198763
3,47115,565380,0.390377
4,27721,332652,0.438525
5,54582,654984,0.320991
6,11178,134136,0.682060
7,49394,592728,0.497618
8,39876,478512,0.196384
9,17374,208488,0.162141


In [48]:
subset = set(df5.columns)
sprset = set(df6.columns)

print(subset.issubset(sprset))
missing_cols = subset - sprset
print("Removed features: ", missing_cols)
added_features = sprset - subset
print("Added features: ", added_features)

True
Removed features:  set()
Added features:  {'income_variance', 'monthly_income_avg', 'annual_income'}


## 📊 Seasonality & Event Features

In [49]:
df7 = df6.copy()

In [50]:
# simulate seasonal workers
seasonal_flag = np.random.choice([0,1], size=len(df7), p=[0.6,0.4])

df7["seasonality_score"] = np.where(seasonal_flag==1,
                                  np.random.uniform(0.6,1.0,len(df7)),
                                  np.random.uniform(0.0,0.4,len(df7)))

df7["income_concentration"] = np.where(seasonal_flag==1,
                                      np.random.uniform(0.5,0.9,len(df7)),
                                      np.random.uniform(0.1,0.4,len(df7)))

df7["event_dependence_score"] = df7["seasonality_score"]

In [51]:
df7.columns

Index(['housing', 'loan', 'job_stability_score', 'marital_divorced',
       'marital_married', 'marital_single', 'education_primary',
       'education_secondary', 'education_tertiary', 'education_unknown',
       'balance_norm', 'monthly_income_avg', 'annual_income',
       'income_variance', 'seasonality_score', 'income_concentration',
       'event_dependence_score'],
      dtype='object')

In [52]:
seasonal_features = [
    'seasonality_score',
    'income_concentration',
    'event_dependence_score'
]
df7[seasonal_features].head(10)

,seasonality_score,income_concentration,event_dependence_score
0,0.240459,0.151987,0.240459
1,0.965127,0.606136,0.965127
2,0.741710,0.804369,0.741710
3,0.277254,0.112012,0.277254
4,0.300026,0.240820,0.300026
5,0.899760,0.621211,0.899760
6,0.328775,0.101220,0.328775
7,0.231489,0.100804,0.231489
8,0.177941,0.351862,0.177941
9,0.381671,0.166823,0.381671


In [53]:
subset = set(df6.columns)
sprset = set(df7.columns)

print(subset.issubset(sprset))
missing_cols = subset - sprset
print("Removed features: ", missing_cols)
added_features = sprset - subset
print("Added features: ", added_features)

True
Removed features:  set()
Added features:  {'seasonality_score', 'income_concentration', 'event_dependence_score'}


## 🧾 Behavioral Features

In [54]:
df8 = df7.copy()

In [55]:
df8["bill_payment_score"] = np.random.uniform(0,1,len(df8))
df8["upi_txn_count"] = np.random.randint(10,300,len(df8))

df8["ecommerce_spend"] = np.random.randint(500,20000,len(df8))
df8["ecommerce_spend_ratio"] = df8["ecommerce_spend"] / df8["monthly_income_avg"]

In [56]:
print(df8.columns)

Index(['housing', 'loan', 'job_stability_score', 'marital_divorced',
       'marital_married', 'marital_single', 'education_primary',
       'education_secondary', 'education_tertiary', 'education_unknown',
       'balance_norm', 'monthly_income_avg', 'annual_income',
       'income_variance', 'seasonality_score', 'income_concentration',
       'event_dependence_score', 'bill_payment_score', 'upi_txn_count',
       'ecommerce_spend', 'ecommerce_spend_ratio'],
      dtype='object')


In [57]:
behavioral_features = [
    'bill_payment_score',
    'upi_txn_count',
    'ecommerce_spend',
    'ecommerce_spend_ratio'
]
df8[behavioral_features].sample(7)

,bill_payment_score,upi_txn_count,ecommerce_spend,ecommerce_spend_ratio
8525,0.556514,95,17778,0.788906
4873,0.966837,187,6825,0.327983
6866,0.275135,285,16655,0.375180
5363,0.023269,73,18385,1.444453
4108,0.772333,47,6105,0.149728
9831,0.016229,53,17614,0.660963
2676,0.549175,188,5444,0.292767


In [58]:
df8.drop(columns=["ecommerce_spend"] ,inplace=True)

In [59]:
subset = set(df7.columns)
sprset = set(df8.columns)

print(subset.issubset(sprset))
missing_cols = subset - sprset
print("Removed features: ", missing_cols)
added_features = sprset - subset
print("Added features: ", added_features)

True
Removed features:  set()
Added features:  {'ecommerce_spend_ratio', 'upi_txn_count', 'bill_payment_score'}


## 📍Location Stability

In [60]:
df9 = df8.copy()

In [61]:
df9["location_stability"] = np.random.uniform(0,1,len(df9))

In [62]:
df9["location_stability"].sample(10)

9926     0.536463
8246     0.794652
8094     0.211040
7708     0.956407
10931    0.060071
7915     0.417239
7225     0.378274
8214     0.070056
1192     0.571728
1596     0.820050
Name: location_stability, dtype: float64

In [63]:
subset = set(df8.columns)
sprset = set(df9.columns)

print(subset.issubset(sprset))
missing_cols = subset - sprset
print("Removed features: ", missing_cols)
added_features = sprset - subset
print("Added features: ", added_features)

True
Removed features:  set()
Added features:  {'location_stability'}


🧠 Psychometric (disabled now)

In [64]:
df10 = df9.copy()

In [65]:
# df["questionnaire_score"] = np.random.uniform(0,1,len(df))

In [66]:
subset = set(df9.columns)
sprset = set(df10.columns)

print(subset.issubset(sprset))
missing_cols = subset - sprset
print("Removed features: ", missing_cols)
added_features = sprset - subset
print("Added features: ", added_features)

True
Removed features:  set()
Added features:  set()


## ⭐ Trust

In [67]:
df11 = df10.copy()

In [68]:
df11["merchant_rating"] = np.random.uniform(1,5,len(df11))

In [69]:
df11["merchant_rating"].head(10)

0    4.390732
1    4.335910
2    2.213710
3    3.697664
4    2.408150
5    4.061447
6    4.185625
7    2.704804
8    2.875219
9    2.226294
Name: merchant_rating, dtype: float64

In [70]:
subset = set(df10.columns)
sprset = set(df11.columns)

print(subset.issubset(sprset))
missing_cols = subset - sprset
print("Removed features: ", missing_cols)
added_features = sprset - subset
print("Added features: ", added_features)

True
Removed features:  set()
Added features:  {'merchant_rating'}


# 🎯 Target

In [71]:
df_final = df11.copy()

In [72]:
print(df_final.columns)

Index(['housing', 'loan', 'job_stability_score', 'marital_divorced',
       'marital_married', 'marital_single', 'education_primary',
       'education_secondary', 'education_tertiary', 'education_unknown',
       'balance_norm', 'monthly_income_avg', 'annual_income',
       'income_variance', 'seasonality_score', 'income_concentration',
       'event_dependence_score', 'bill_payment_score', 'upi_txn_count',
       'ecommerce_spend_ratio', 'location_stability', 'merchant_rating'],
      dtype='object')


#### Education score.

In [73]:
df_final["education_score"] = (
    0.2 * df_final["education_primary"] +
    0.4 * df_final["education_secondary"] +
    0.7 * df_final["education_tertiary"] +
    0.3 * df_final["education_unknown"]
)

#### Marital Score

In [74]:
df_final["marital_score"] = (
    0.6 * df_final["marital_married"] +
    0.4 * df_final["marital_single"] +
    0.5 * df_final["marital_divorced"]
)

#### Seasonal bonus

In [75]:
seasonal_bonus = (
    df_final["seasonality_score"] *
    df_final["event_dependence_score"]
)

## Risk Calculation

In [76]:
risk = (
    0.18 * df_final["income_variance"] * (1 - df_final["seasonality_score"]) +
    0.18 * (1 - df_final["bill_payment_score"]) +
    0.14 * (1 - df_final["location_stability"]) +
    0.12 * df_final["ecommerce_spend_ratio"] +
    0.12 * (df_final["income_concentration"] * (1 - df_final["event_dependence_score"])) +
    0.10 * (1 - df_final["job_stability_score"]) +
    0.08 * (1 - df_final["balance_norm"]) +
    0.04 * df_final["loan"] +
    0.04 * df_final["housing"] +
    0.04 * (1 - df_final["marital_score"]) +
    0.04 * (1 - df_final["education_score"])
)

# seasonal correction
risk = risk - 0.10 * seasonal_bonus

# keep within bounds
risk = risk.clip(0, 1)

## Target

In [77]:
df_final["default"] = (risk > 0.5).astype(int)

## Output

In [78]:
print(risk.describe())
print(df_final["default"].value_counts())

count    11162.000000
mean         0.405004
std          0.106026
min          0.060500
25%          0.331453
50%          0.404705
75%          0.476539
max          0.904982
dtype: float64
default
0    9101
1    2061
Name: count, dtype: int64


In [79]:
for column in list(df_final.columns):
    print(column)

housing
loan
job_stability_score
marital_divorced
marital_married
marital_single
education_primary
education_secondary
education_tertiary
education_unknown
balance_norm
monthly_income_avg
annual_income
income_variance
seasonality_score
income_concentration
event_dependence_score
bill_payment_score
upi_txn_count
ecommerce_spend_ratio
location_stability
merchant_rating
education_score
marital_score
default


In [80]:
print(df_final.shape)

(11162, 25)


In [81]:
# import packages.
import pandas as pd
import numpy as np

In [82]:
'''Exported dataset files.'''
# df.to_excel("bank1.xlsx", index=False)
# df.to_excel("bank2.xlsx", index=False)
# df.to_excel("bank3.xlsx", index=False)
df_final.to_csv("bank_final.csv", index=False)